In [1]:
import json
import os
import glob
import shutil
import sys
from typing import Dict, List

from logger import _setup_child_logger, _setup_root_logger
from synthetic_metrics_plotter import (
    SyntheticMetricsPlotter,
)

# Set up the root logger and module level logger. The module level logger is
# a child of the root logger.
_setup_root_logger()
logger = _setup_child_logger(__name__)


In [2]:
class SyntheticPlotsParameters:
    def __init__(
            self,
            case_name: str = "cmip6.historical",
            model_prefix: str = "e3sm.historical.v3-LR",
            model_pattern: str = "v3.LR.historical",
            cmip_clim_set: str = "cmip6.historical.v20250707",
            cmip_movs_set: str = "cmip6.historical.v20220825", 
            cmip_enso_set: str = "cmip6.historical.v20210620",
            clim_vars: str = "pr,prw,psl,rlds,rldscs,rltcre,rstcre,rsus,rsuscs,rlus",
            clim_regions: str = "global,ocean,land",
            test_mean_only: bool = False,
            movs_group: str = "cbf",
            error_norm: str = "default", 
            metric_mean_list: list = None, 
            exclude_vars: dict = None,
            exclude_models: list = None, 
            atm_modes: str = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2",
            atm_obs: str = "NOAA-20C",
            cpl_modes: str = "PDO,NPGO,AMO", 
            cpl_obs: str = "HadISST",
            diag_path: str = "./", 
            data_dir: str = "/lcrc/group/e3sm/www/pcmdi", 
            out_dir: str = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang",
            run_type: str = "model_vs_obs",
            case_id: str = "v20250929",
            save_all_data: bool = True,
            figure_format: str = "png",
            clim_viewer: bool = True,
            mova_viewer: bool = True,
            movc_viewer: bool = True,
            enso_viewer: bool = True,
            mean_group1_name: str = "CMIP",
            mean_group2_name: str = "E3SM",
            extra_groups_name: list = None,
        ):
        
        """Initialize default parameters for PCMDI synthetic plots."""
        self.figure_format = figure_format
        self.save_all_data = save_all_data
        self.run_type = run_type
        self.model_tableID = "Amon"
        self.data_dir = data_dir
        self.out_dir = out_dir

        # === Discover models ===
        self.case = case_name # group name 
        pattern = os.path.join(data_dir, f"{model_pattern}_*")
        self.model_name = sorted([os.path.basename(d) for d in glob.glob(pattern) if os.path.isdir(d)])
        
        if enso_viewer and (not clim_viewer or not mova_viewer or not movc_viewer):
            self.model_name = [
                m for m in self.model_name
                if not any(tag in m for tag in ("0131", "0241"))
            ]
            
        self.test_name = [
            f"{model_prefix}.{name.split(f'{model_pattern}_')[-1]}"
            for name in self.model_name
        ]
        self.test_mean_only = test_mean_only 
        self.movs_group = movs_group
        self.error_norm = error_norm
        self.case_id = [case_id] * len(self.model_name)
        self.exclude_vars = exclude_vars 
        self.exclude_models = exclude_models
        self.mean_group1_name = mean_group1_name
        self.mean_group2_name = mean_group2_name
        self.extra_groups_name = extra_groups_name
        
        # === CMIP reference sets ===
        self.cmip_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
        self.cmip_clim_set = cmip_clim_set
        self.cmip_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
        self.cmip_movs_set = cmip_movs_set
        self.cmip_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
        self.cmip_enso_set = cmip_enso_set

        # === Variable sets ===
        self.clim_vars = clim_vars.split(",")
        self.clim_regions = clim_regions.split(",")
        self.mova_modes = atm_modes.split(",")
        self.movc_modes = cpl_modes.split(",")
        self.mova_obs = atm_obs
        self.movc_obs = cpl_obs
        
        # === Viewer toggles ===
        self.clim_viewer = clim_viewer
        self.mova_viewer = mova_viewer
        self.movc_viewer = movc_viewer
        self.enso_viewer = enso_viewer

        # === Metadata ===
        self.pcmdi_webtitle = "PCMDI Synthetic Metrics"
        self.pcmdi_version = "v1"
        self.pcmdi_external_prefix = ""
        self.pcmdi_viewer_template = ""

        # === Derived paths ===
        self.base_test_input_path = os.path.join(
            self.data_dir,
            "%(model_name)",
            "pcmdi_diags",
            self.run_type,
            "metrics_data",
            "%(group_type)",
        )

        self.results_dir_full = os.path.join(self.out_dir, self.run_type)

    def summary(self):
        print(f"Case: {self.case}")
        print(f"Models ({len(self.model_name)}): {self.model_name}")
        print(f"Results → {self.results_dir_full}")
        
    def str2bool(self, v):
        if isinstance(v, bool):
            return v
        val = str(v).lower()
        if val in ("yes", "true", "t", "1", "y", "on"):
            return True
        elif val in ("no", "false", "f", "0", "n", "off"):
            return False
        else:
            raise argparse.ArgumentTypeError(f"Invalid boolean value: {v}")
            
    def load_metrics(self, metric_file='synthetic_metrics_list.json'):
        """Load metrics from the JSON file."""
        try:
            with open(metric_file) as f:
                return json.load(f)
        except FileNotFoundError as e:
            logging.error(f"File not found: {e}")
            raise
        except json.JSONDecodeError as e:
            logging.error(f"Error decoding JSON: {e}")
            raise

In [3]:
if __name__ == "__main__":
    """Main function to drive the entire process."""
    # Configuration
    run_type = 'model_vs_obs'
    diag_path = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le"
    data_path = os.path.join(diag_path,'climo')
    out_path = os.path.join(diag_path,'Climatology_Metrics')

    metric_sets = 'mean_climate,variability_modes,enso_metric'.split(",")
    figure_sets = 'portrait,parcoord'.split(",")

    case_name = "E3SMv3-LE-CLIM" 
    model_prefix = "e3sm.historical.v3-LR"
    model_pattern = "v3.LR.historical"
    
    cmip_clim_set = None #"cmip6.historical.v20250707"
    cmip_movs_set = None #"cmip6.historical.v20220825"
    cmip_enso_set = None #"cmip6.historical.v20210620"
    
    atm_modes = "PSA1,PSA2"
    cpl_modes = "AMO"
    
    #atm_modes = "NAM,NAO,PNA,NPO,SAM"
    #cpl_modes = "PDO,NPGO"
    
    movs_group = "cbf" #"eof"
    atm_obs = "NOAA-20C"
    cpl_obs = "HadISST"
    
    clim_vars = "pr,prw,psl,rlds,rldscs,rltcre,rstcre,rsus,rsuscs,rlus,rlut,rlutcs,rsds,rsdscs,rsut,rsutcs,rtmt,sfcWind,tas,tauu,tauv,ts,ta-200,ta-850,ua-200,ua-850,va-200,va-850,zg-500"
    clim_regions = "global"
    
    figure_format = "png"
    save_all_data = True 
    test_mean_only = False
    error_norm = "reference" # or "default" (selected reference)
    
    clim_viewer = False
    mova_viewer = True
    movc_viewer = True
    enso_viewer = False #True
    case_id = "v20251014" 

    ################################
    if clim_viewer: 
        case_id = "v20251014" 
        if test_mean_only:
            out_dir = f"{out_path}/clim_vs_{error_norm}_mean"
        else:
            out_dir = f"{out_path}/clim_vs_{error_norm}_all"
            
    if mova_viewer or movc_viewer:
        test_mean_only = False
        error_norm = "reference"
        case_id = "v20251014"
        out_dir = f"{out_path}/movs_{atm_obs}_{cpl_obs}_{movs_group}"
        
    if enso_viewer:
        test_mean_only = False
        error_norm = "reference"
        case_id = "v20251014"
        out_dir = f"{out_path}/enso_with_feedback"
        
    mean_group1_name = "E3SMv3-LR" 
    mean_group2_name = "E3SMv3-LR"
    extra_groups_name = None

    ##################################
    exclude_vars = {
        "E3SM-1-0":     ["ta-850"],
        "E3SM-1-1-ECA": ["ta-850"],
        "CIESM":        ["pr"],
        "KIOST-ESM":    ["zg-500", "ta-850"],
        "GISS-E2-2-G":  ["rlutcs","zg-500"],
        
    }
    
    exclude_models = ['E3SM-1-0','E3SM-1-1','E3SM-1-1-ECA','E3SM-2-0','E3SM-2-1']
    
    parameters = SyntheticPlotsParameters(
        case_name = case_name,
        model_prefix = model_prefix,
        model_pattern = model_pattern,
        cmip_clim_set = cmip_clim_set,
        cmip_movs_set = cmip_movs_set, 
        cmip_enso_set = cmip_enso_set,        
        clim_vars = clim_vars,
        clim_regions = clim_regions,
        test_mean_only = test_mean_only,
        movs_group = movs_group,
        error_norm = error_norm, 
        exclude_vars = exclude_vars,
        exclude_models = exclude_models,
        atm_modes = atm_modes,
        atm_obs = atm_obs,
        cpl_modes = cpl_modes,
        cpl_obs = cpl_obs,
        diag_path = diag_path, 
        data_dir = data_path, 
        out_dir  = out_dir, 
        run_type = run_type,
        case_id = case_id,        
        save_all_data = save_all_data,
        figure_format = figure_format,
        clim_viewer = clim_viewer,
        mova_viewer = mova_viewer,
        movc_viewer = movc_viewer,
        enso_viewer = enso_viewer,
    )
    
    print(parameters.model_name)
    print(parameters.case_id)
    print(parameters.case_id)

    #########################################
    # plot synthetic figures for pcmdi metrics
    #########################################
    logger.info("generate synthetic metrics plot ...")
    test_input_path = os.path.join(
        parameters.data_dir,
        "put_model_here",
        "pcmdi_diags",
        parameters.run_type,
        "metrics_data",
        "%(group_type)",
    )
    
    metric_dict = json.load(open("synthetic_metrics_list.json"))

    plotter = SyntheticMetricsPlotter(
        # Core
        model_name=parameters.model_name,
        test_name=parameters.test_name,
        case_id=parameters.case_id, 
        table_id=parameters.model_tableID,
        figure_format=parameters.figure_format,
        metric_dict=metric_dict,
        save_data=parameters.save_all_data,
        base_test_input_path=test_input_path,
        results_dir=parameters.results_dir_full,
        # Mean climate
        clim_viewer=parameters.clim_viewer,
        clim_vars=parameters.clim_vars,
        clim_regions=parameters.clim_regions,
        cmip_clim_dir=parameters.cmip_clim_dir,
        cmip_clim_set=parameters.cmip_clim_set,
        # MOVA
        mova_viewer=parameters.mova_viewer,
        mova_modes=parameters.mova_modes,
        mova_obs=parameters.mova_obs,
        # MOVC
        movc_viewer=parameters.movc_viewer,
        movc_modes=parameters.movc_modes,
        movc_obs=parameters.movc_obs,
        cmip_movs_dir=parameters.cmip_movs_dir,
        cmip_movs_set=parameters.cmip_movs_set,
        # ENSO
        enso_viewer=parameters.enso_viewer,
        cmip_enso_dir=parameters.cmip_enso_dir,
        cmip_enso_set=parameters.cmip_enso_set,
        # Setup for visualization
        test_mean_only = parameters.test_mean_only,
        movs_group = parameters.movs_group,
        exclude_vars = parameters.exclude_vars,
        exclude_models = parameters.exclude_models, 
        error_norm = parameters.error_norm, 
        mean_group1_name = mean_group1_name,
        mean_group2_name = mean_group2_name,
        extra_groups_name = extra_groups_name,
    )

    # Generate Summary Metrics plots
    # e.g., "climatology,enso,variability"
    figure_sets = []
    if parameters.clim_viewer:
        figure_sets.append("climatology")
    if parameters.mova_viewer:
        figure_sets.append("variability(ATM)")
    if parameters.movc_viewer:
        figure_sets.append("variability(CPL)")
    if parameters.enso_viewer:
        figure_sets.append("enso")

    logger.info(f"Generating groups={figure_sets}")
    # This calls the _handle_{figure_set} functions
    # Those call the {figure_set}_plot_driver functions
    plotter.generate()
    

2026-01-31 00:58:28,143 [INFO]: 1296512855.py(<module>:118) >> generate synthetic metrics plot ...


['v3.LR.historical_0051', 'v3.LR.historical_0091', 'v3.LR.historical_0101', 'v3.LR.historical_0111', 'v3.LR.historical_0121', 'v3.LR.historical_0131', 'v3.LR.historical_0141', 'v3.LR.historical_0151', 'v3.LR.historical_0161', 'v3.LR.historical_0171', 'v3.LR.historical_0181', 'v3.LR.historical_0191', 'v3.LR.historical_0201', 'v3.LR.historical_0211', 'v3.LR.historical_0221', 'v3.LR.historical_0231', 'v3.LR.historical_0241', 'v3.LR.historical_0251', 'v3.LR.historical_0261', 'v3.LR.historical_0271', 'v3.LR.historical_0281', 'v3.LR.historical_0291', 'v3.LR.historical_0301', 'v3.LR.historical_0311', 'v3.LR.historical_0321']
['v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014', 'v20251014']
['v20251014', 'v20251014', 'v20251014', 'v202510

TypeError: SyntheticMetricsPlotter.__init__() got an unexpected keyword argument 'model_name'